# MiniMax H3 · A100 80GB

选择 A100 80GB，依次运行：**环境 → 模型与工作流 → 打开 ComfyUI → 下载结果**。

01 文生视频；02 首帧；03 首尾帧；04 双图参考。图片版先上传图片，比例自动跟随首帧或参考图 1。默认基础模式、20 步、5 秒。

第 2 区下载工作流，同名直接覆盖。第 3 区点击新窗口链接并保持 Colab 连接；生成后在第 4 区下载单个或全部视频。04 尚未提供音频、视频上传入口。


In [ ]:
# @title 1.1 环境：检查 A100 / CUDA
import subprocess, torch

print(subprocess.run(
    ["nvidia-smi"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
).stdout)

assert torch.cuda.is_available(), "没有检测到 CUDA GPU。"

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3

print("显卡：", gpu_name)
print(f"显存： {vram_gb:.1f} GB")
print("PyTorch 版本：", torch.__version__)
print("CUDA 版本：", torch.version.cuda)

if "A100" in gpu_name.upper() and vram_gb >= 70:
    print("[OK] A100 80GB 环境正常。")
else:
    print("[WARN] 当前环境不是目标 A100 80GB 配置。")

In [ ]:
# @title 1.2 环境：安装 / 更新 ComfyUI + 依赖
import os, re, sys, pathlib, subprocess

COMFY_DIR = "/content/ComfyUI"

def run(cmd, cwd=None):
    print("+", " ".join(cmd))
    subprocess.check_call(cmd, cwd=cwd)

if not os.path.isdir(COMFY_DIR):
    run([
        "git", "clone", "--depth", "1",
        "https://github.com/Comfy-Org/ComfyUI.git",
        COMFY_DIR
    ])
else:
    run(["git", "fetch", "--depth", "1", "origin", "master"], cwd=COMFY_DIR)
    run(["git", "reset", "--hard", "origin/master"], cwd=COMFY_DIR)

req = pathlib.Path(COMFY_DIR) / "requirements.txt"
filtered = pathlib.Path("/content/comfy_requirements_no_runtime_torch.txt")
skip_exact = {"torch", "torchvision", "torchaudio", "xformers"}

kept = []
for line in req.read_text().splitlines():
    s = line.strip()
    if not s or s.startswith("#"):
        kept.append(line)
        continue
    name = re.split(r"[<>=!~\[]", s, maxsplit=1)[0].strip().lower()
    if name in skip_exact:
        print("保留 Colab 运行时版本，跳过：", s)
        continue
    kept.append(line)

filtered.write_text("\n".join(kept) + "\n")

run([sys.executable, "-m", "pip", "install", "-q", "-r", str(filtered)])
run([sys.executable, "-m", "pip", "install", "-q", "-U",
     "torchsde", "huggingface_hub", "hf_xet", "ipywidgets"])

sha = subprocess.check_output(
    ["git", "rev-parse", "--short", "HEAD"],
    cwd=COMFY_DIR, text=True
).strip()
print("ComfyUI 版本：", sha)

In [ ]:
# @title 1.3 环境：使用 /content 本地模型和输出目录
import os

COMFY_DIR = "/content/ComfyUI"
MODEL_SUBDIRS = ["diffusion_models", "text_encoders", "vae", "loras", "embeddings"]

for sub in MODEL_SUBDIRS:
    p = os.path.join(COMFY_DIR, "models", sub)

    # 清掉旧版本可能留下的 Google Drive symlink，但不会删除 Drive 文件本身。
    if os.path.islink(p):
        print("移除软链接:", p, "->", os.path.realpath(p))
        os.unlink(p)

    os.makedirs(p, exist_ok=True)
    print(p)
    print("  是否为软链接：", os.path.islink(p))
    print("  实际路径：", os.path.realpath(p))

# 继续使用旧运行时时，一并解除旧的 output 软链接。
# 只移除软链接，不会删除 Drive 中已经生成的视频。
output_dir = os.path.join(COMFY_DIR, "output")
if os.path.islink(output_dir):
    print("移除输出软链接:", output_dir, "->", os.path.realpath(output_dir))
    os.unlink(output_dir)
os.makedirs(output_dir, exist_ok=True)
print(output_dir)
print("  是否为软链接：", os.path.islink(output_dir))
print("  实际路径：", os.path.realpath(output_dir))

print("[OK] 模型和输出目录使用 Colab 本地磁盘。")

In [ ]:
# @title 2. 模型与 GitHub 工作流
DOWNLOAD_R2V = True # @param {type:"boolean"}
DOWNLOAD_TURBO_LORAS = True # @param {type:"boolean"}

# Turbo LoRA 默认下载是为了让官方模板不报 missing model；
# 实际生成仍建议先保持 turbo_mode=False / 20 steps。

import os
from huggingface_hub import hf_hub_download

COMFY_DIR = "/content/ComfyUI"
MODELS_DIR = os.path.join(COMFY_DIR, "models")
H3_REPO = "Comfy-Org/MiniMax-H3"
TURBO_REPO = "lightx2v/Minimax-h3-Turbo"

downloads = [
    (H3_REPO, "diffusion_models/minimax_h3_fl2va_pruned_int8_convrot.safetensors", "diffusion_models"),
    (H3_REPO, "text_encoders/qwen3vl_32b_minimax_h3_int8_convrot.safetensors", "text_encoders"),
    (H3_REPO, "vae/minimax_h3_video_vae_fp16.safetensors", "vae"),
    (H3_REPO, "vae/minimax_h3_audio_vae_fp32.safetensors", "vae"),
]

if DOWNLOAD_R2V:
    downloads.append((
        H3_REPO,
        "diffusion_models/minimax_h3_ref2va_pruned_int8_convrot.safetensors",
        "diffusion_models"
    ))

if DOWNLOAD_TURBO_LORAS:
    downloads += [
        (
            TURBO_REPO,
            "minimax_h3_fl2v_turbo_8step_v1.0_comfyui_bf16.safetensors",
            "loras"
        ),
        (
            H3_REPO,
            "loras/minimax_h3_ref2v_turbo_4step_v0.1_comfyui_bf16.safetensors",
            "loras"
        )
    ]

for repo_id, remote_path, group in downloads:
    if repo_id == H3_REPO:
        target = os.path.join(MODELS_DIR, remote_path)
    else:
        target = os.path.join(MODELS_DIR, group, os.path.basename(remote_path))

    os.makedirs(os.path.dirname(target), exist_ok=True)

    if os.path.isfile(target) and os.path.getsize(target) > 1_000_000:
        print("已缓存：", os.path.basename(target),
              f"{os.path.getsize(target)/1024**3:.2f} GB")
        continue

    print("\n正在下载：", repo_id, remote_path)

    if repo_id == H3_REPO:
        hf_hub_download(
            repo_id=repo_id,
            filename=remote_path,
            local_dir=MODELS_DIR
        )
    else:
        hf_hub_download(
            repo_id=repo_id,
            filename=remote_path,
            local_dir=os.path.join(MODELS_DIR, group)
        )

print("\n[OK] 模型准备完成。")
WORKFLOW_REF = "main" # @param {type:"string"}

import json, os, tempfile, urllib.error, urllib.request
from pathlib import Path
from urllib.parse import quote, urlencode

WORKFLOW_REPO = "boltguo/colab"
WORKFLOW_FOLDER = "workflows/minimax-h3"
WORKFLOW_DIR = Path(COMFY_DIR) / "user/default/workflows"
MODEL_DIR = Path(MODELS_DIR)
EXPECTED_LOADERS = {
    "UNETLoader": ("diffusion_models", {
        "minimax_h3_fl2va_pruned_int8_convrot.safetensors",
        "minimax_h3_ref2va_pruned_int8_convrot.safetensors"}),
    "CLIPLoader": ("text_encoders", {"qwen3vl_32b_minimax_h3_int8_convrot.safetensors"}),
    "VAELoader": ("vae", {"minimax_h3_video_vae_fp16.safetensors", "minimax_h3_audio_vae_fp32.safetensors"}),
    "LoraLoaderModelOnly": ("loras", {
        "minimax_h3_fl2v_turbo_8step_v1.0_comfyui_bf16.safetensors",
        "minimax_h3_ref2v_turbo_4step_v0.1_comfyui_bf16.safetensors"}),
}

def local_workflow_directory(path):
    path = Path(path)
    for parent in [path, *path.parents]:
        if parent == Path(COMFY_DIR).parent:
            break
        if parent.is_symlink():
            raise ValueError(f"工作流目录是软链接，未写入：{parent}")
    path.mkdir(parents=True, exist_ok=True)

def workflow_nodes(data):
    if not isinstance(data, dict) or not isinstance(data.get("nodes"), list) or not data["nodes"] or not isinstance(data.get("links"), list):
        raise ValueError("工作流 JSON 缺少 nodes 或 links。")
    definitions = data.get("definitions", {})
    if not isinstance(definitions, dict) or not isinstance(definitions.get("subgraphs", []), list):
        raise ValueError("工作流 definitions/subgraphs 结构无效。")
    for graph in [data, *definitions.get("subgraphs", [])]:
        if not isinstance(graph, dict) or not isinstance(graph.get("nodes"), list) or not isinstance(graph.get("links"), list):
            raise ValueError("工作流或子图需要 nodes 和 links。")
        for node in graph["nodes"]:
            if not isinstance(node, dict) or not isinstance(node.get("type"), str):
                raise ValueError("工作流节点结构无效。")
            yield node

def validate_workflow_models(data):
    for node in workflow_nodes(data):
        expected = EXPECTED_LOADERS.get(node["type"])
        values = node.get("widgets_values") or []
        if expected and (not isinstance(values, list) or not values or values[0] not in expected[1]):
            raise ValueError(f"{node['type']} 的模型文件名与本 notebook 下载映射不一致。")
    return data

def report_model_requirements(name, data):
    for node in workflow_nodes(data):
        expected = EXPECTED_LOADERS.get(node["type"])
        if not expected:
            continue
        filename = node["widgets_values"][0]
        target = MODEL_DIR / expected[0] / filename
        if target.is_file() and not target.is_symlink():
            continue
        if "ref2va" in filename:
            print(f"前置要求：{name} 需要 Ref2VA。请启用 DOWNLOAD_R2V 并重跑本格：{target}。不会用 FL2VA 替代。")
        elif expected[0] == "loras":
            print(f"前置要求：{name} 保留加速分支；请启用 DOWNLOAD_TURBO_LORAS 准备 {filename}。默认使用基础模式；切换加速模式前确认权重已下载。")
        else:
            print(f"前置要求：{name} 的模型缺失，请重跑本格：{target}")

def github_json(url):
    request = urllib.request.Request(url, headers={"User-Agent": "MiniMax-Colab-Workflow-Downloader", "Accept": "application/vnd.github+json"})
    with urllib.request.urlopen(request, timeout=30) as response:
        body = response.read(2_000_001)
        if len(body) > 2_000_000 or "text/html" in response.headers.get("Content-Type", "").lower():
            raise ValueError("GitHub 返回网页或文件超过 2 MB，未保存。")
    try:
        return json.loads(body)
    except (UnicodeDecodeError, json.JSONDecodeError) as error:
        raise ValueError("GitHub 返回内容不是 JSON；未保存。") from error

def install_workflow(name, data):
    if not isinstance(name, str) or not name.lower().endswith(".json") or Path(name).name != name or "\\" in name:
        raise ValueError("工作流文件名无效。")
    validate_workflow_models(data)
    local_workflow_directory(WORKFLOW_DIR)
    target = WORKFLOW_DIR / name
    if target.is_symlink() or (target.exists() and not target.is_file()):
        raise ValueError("目标工作流是软链接或目录，未写入。")
    fd, temporary = tempfile.mkstemp(prefix=".minimax-workflow-", dir=WORKFLOW_DIR)
    try:
        with os.fdopen(fd, "w", encoding="utf-8") as file:
            json.dump(data, file, ensure_ascii=False, indent=2)
            file.write("\n")
        os.replace(temporary, target)
        print("工作流已下载：", name)
        report_model_requirements(name, data)
        return True
    finally:
        if os.path.exists(temporary):
            os.unlink(temporary)

def report_github_failure(error, name=None):
    label = f"工作流 {name}" if name else "GitHub 工作流目录"
    if isinstance(error, urllib.error.HTTPError) and error.code == 404:
        print(f"{label} 返回 404：可能尚未推送到 GitHub，或 WORKFLOW_REF={WORKFLOW_REF!r} 不存在。推送后重跑‘2. 模型与 GitHub 工作流’；保留当前工作流。")
    else:
        print(f"{label} 下载失败：{error}。未保存错误页面，保留当前工作流。")

def download_github_workflows():
    ref = WORKFLOW_REF.strip()
    if not ref:
        raise ValueError("WORKFLOW_REF 不能为空。")
    api = f"https://api.github.com/repos/{WORKFLOW_REPO}/contents/{WORKFLOW_FOLDER}?" + urlencode({"ref": ref})
    try:
        entries = github_json(api)
        if not isinstance(entries, list):
            raise ValueError("GitHub 目录响应不是文件列表。")
    except (urllib.error.URLError, OSError, ValueError) as error:
        report_github_failure(error)
        return
    entries = [entry for entry in entries if isinstance(entry, dict) and entry.get("type") == "file" and str(entry.get("name", "")).lower().endswith(".json")]
    if not entries:
        print("该 GitHub 目录目前没有 JSON 工作流；推送后重跑‘2. 模型与 GitHub 工作流’。")
    for entry in sorted(entries, key=lambda entry: entry["name"]):
        name = entry["name"]
        try:
            if not isinstance(name, str) or Path(name).name != name or "\\" in name:
                raise ValueError("GitHub 文件名无效。")
            url = f"https://raw.githubusercontent.com/{WORKFLOW_REPO}/{quote(ref, safe='')}/{WORKFLOW_FOLDER}/{quote(name, safe='')}"
            install_workflow(name, github_json(url))
        except (urllib.error.URLError, OSError, ValueError) as error:
            report_github_failure(error, name)

local_workflow_directory(WORKFLOW_DIR)
download_github_workflows()
print("在 ComfyUI 工作流列表打开 01–04；02/03/04 需要选择自己的参考输入。")


In [ ]:
# @title 3. 启动并打开 ComfyUI
RESTART_COMFYUI = True # @param {type:"boolean"}

import os, signal, subprocess, sys, time, urllib.request

COMFY_DIR = "/content/ComfyUI"
BASE = "http://127.0.0.1:8188"
LOG_PATH = "/content/comfyui_h3.log"
PID_PATH = "/content/comfyui_h3.pid"

from pathlib import Path
LAUNCHER = Path("/content/minimax_comfy_userdata_compat.py")
LAUNCHER_SOURCE = '\n"""Start pinned ComfyUI with narrow workflow aliases for decoded proxy paths."""\nimport os\nimport runpy\nimport sys\nfrom pathlib import Path\nfrom aiohttp import web\n\n\ndef add_workflow_aliases(manager, routes):\n    originals = [route for route in routes\n                 if getattr(route, "path", None) == "/userdata/{file}"\n                 and getattr(route, "method", None) in {"GET", "POST", "DELETE"}]\n    if {route.method for route in originals} != {"GET", "POST", "DELETE"}:\n        raise RuntimeError("ComfyUI 工作流访问接口已变化，未安装兼容路由。")\n    for route in originals:\n        async def workflow_alias(request, handler=route.handler):\n            file = request.match_info.get("file", "")\n            user_root = manager.get_request_user_filepath(request, None, create_dir=False)\n            candidate = manager.get_request_user_filepath(request, file, create_dir=False)\n            if not user_root or not candidate:\n                return web.Response(status=403)\n            workflow_directory = os.path.join(user_root, "workflows")\n            if os.path.islink(workflow_directory):\n                return web.Response(status=403)\n            workflow_root = os.path.realpath(workflow_directory)\n            try:\n                within = os.path.commonpath((workflow_root, os.path.realpath(candidate))) == workflow_root\n            except ValueError:\n                within = False\n            if not within or os.path.realpath(candidate) == workflow_root:\n                return web.Response(status=403)\n            # Preserve the original user checks, safe response headers,\n            # atomic writes, overwrite flags, and deletion semantics.\n            return await handler(request)\n        routes.route(route.method, "/userdata/{file:workflows/[^/]+}")(workflow_alias)\n\n\ndef main():\n    comfy_root = Path("/content/ComfyUI")\n    if comfy_root.is_symlink() or not (comfy_root / "main.py").is_file() or not (comfy_root / "comfy/options.py").is_file():\n        raise RuntimeError("ComfyUI 源码目录缺失或无效。")\n    os.chdir(comfy_root)\n    sys.path.insert(0, str(comfy_root))\n    sys.argv[0] = str(comfy_root / "main.py")\n    import comfy.options\n    comfy.options.enable_args_parsing()\n    from app.user_manager import UserManager\n    original = UserManager.add_routes\n\n    def compatible_routes(self, routes):\n        original(self, routes)\n        add_workflow_aliases(self, routes)\n\n    UserManager.add_routes = compatible_routes\n    runpy.run_path("main.py", run_name="__main__")\n\n\nif __name__ == "__main__":\n    main()\n'
LAUNCHER.write_text(LAUNCHER_SOURCE)

def ready():
    try:
        with urllib.request.urlopen(BASE + "/system_stats", timeout=2) as r:
            return r.status == 200
    except Exception:
        return False

def stop_saved_pid():
    if not os.path.exists(PID_PATH):
        return
    try:
        pid = int(open(PID_PATH).read().strip())
        os.kill(pid, signal.SIGTERM)
        print("停止旧 ComfyUI PID:", pid)
        time.sleep(2)
    except Exception:
        pass

if RESTART_COMFYUI:
    stop_saved_pid()

if ready() and not RESTART_COMFYUI:
    print("[OK] ComfyUI 已经在运行。")
else:
    log_fp = open(LOG_PATH, "w")
    proc = subprocess.Popen(
        [
            sys.executable, str(LAUNCHER),
            "--listen", "127.0.0.1",
            "--port", "8188",
            "--enable-cors-header",
        ],
        cwd=COMFY_DIR,
        stdout=log_fp,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )

    with open(PID_PATH, "w") as f:
        f.write(str(proc.pid))

    print("ComfyUI PID:", proc.pid)
    print("log:", LOG_PATH)

    deadline = time.time() + 600
    while time.time() < deadline:
        if proc.poll() is not None:
            print(open(LOG_PATH, errors="replace").read()[-10000:])
            raise RuntimeError(f"ComfyUI 提前退出，code={proc.returncode}")
        if ready():
            print("[OK] ComfyUI 启动成功。")
            break
        time.sleep(3)
    else:
        raise TimeoutError("ComfyUI 10 分钟内没有就绪。")
import html, urllib.request
from google.colab.output import eval_js
from IPython.display import HTML, display

try:
    with urllib.request.urlopen("http://127.0.0.1:8188/system_stats", timeout=5) as r:
        if r.status != 200:
            raise RuntimeError(f"HTTP {r.status}")
except Exception as exc:
    raise RuntimeError("ComfyUI 8188 服务未就绪，请先运行第 3 区，并检查 /content/comfyui_h3.log。") from exc

print("[OK] ComfyUI 服务正常。检查 Colab 网页通信（最多 5 秒）……", flush=True)
try:
    frontend_result = eval_js("1 + 1", timeout_sec=5)
except Exception as exc:
    raise RuntimeError("Colab 网页通信报错，请查看下方原始异常。") from exc

if frontend_result != 2:
    raise TimeoutError(
        "ComfyUI 服务正常，但未收到 Colab 网页的 JavaScript 回复，尚未请求端口代理。"
        "如果通过编辑器或其他 Notebook 客户端执行，请在 Colab 网页执行第 3 区；"
        "如果已在 Colab 网页，请刷新页面并确认已连接后重跑第 3 区。"
    )

print("[OK] Colab 网页通信正常。获取 8188 代理地址（最多 20 秒）……", flush=True)
try:
    # 保留已验证的 Colab 端口代理调用及等待上限。
    url = eval_js("google.colab.kernel.proxyPort(8188)", timeout_sec=20)
except Exception as exc:
    raise RuntimeError("Colab 网页通信正常，但 8188 端口代理报错，请查看下方原始异常。") from exc

if not isinstance(url, str) or not url.strip():
    raise TimeoutError(
        "Colab 网页通信检查已通过，但 8188 代理地址在 20 秒内没有返回。"
        "请保持当前运行时，刷新 Colab 网页并重新连接后，只重跑第 3 区。"
    )

print("ComfyUI:", url)
display(HTML(
    '<p><a href="' + html.escape(url, quote=True) + '" target="_blank" rel="noopener" '
    'style="font-size:20px;font-weight:700">打开 ComfyUI 8188</a></p>'
))
print("打开链接时保持 Colab 页面连接。若链接已生成但出现 403，需要检查代理页面的访问权限或浏览器限制。")

打开工作流 **01–04**，上传所需图片并修改提示词后运行。03 需首尾两张图，04 需两张参考图。

完成后运行第 4 区下载；重跑下载格刷新列表。


In [ ]:
# @title 4. 下载视频与工作流
import json, os, tempfile, zipfile
from pathlib import Path
import av
import ipywidgets as widgets
from IPython.display import display
from google.colab import files

OUTPUT_DIR = Path("/content/ComfyUI/output")
VIDEO_EXTENSIONS = {".mp4", ".webm", ".mov", ".mkv"}

def result_files(directory):
    directory = Path(directory)
    if directory.is_symlink():
        raise ValueError("输出目录是软链接，请核对保存目录。")
    root = directory.resolve()
    results = []
    for base, folders, names in os.walk(directory, followlinks=False):
        folders[:] = [name for name in folders if not (Path(base) / name).is_symlink()]
        for name in names:
            path = Path(base) / name
            if path.suffix.lower() in VIDEO_EXTENSIONS and not path.is_symlink() and path.is_file() and path.resolve().is_relative_to(root):
                results.append(path)
    return sorted(results, key=lambda path: path.stat().st_mtime, reverse=True)

def embedded_workflow(selected):
    try:
        with av.open(str(selected), mode="r") as video:
            metadata = {key.lower(): value for key, value in video.metadata.items()}
        embedded = metadata.get("workflow")
        if embedded:
            data = json.loads(embedded)
            if isinstance(data, dict) and isinstance(data.get("nodes"), list) and isinstance(data.get("links"), list):
                return data
    except Exception as error:
        print("无法读取内嵌工作流，仍下载视频：", error)
    print("视频未内嵌工作流，ZIP 仅含视频。可在 ComfyUI 导出 JSON。")
    return None

def make_videos_zip(selected, directory, archive, include_workflow=False):
    directory, archive = Path(directory), Path(archive)
    available = result_files(directory)
    selected = [Path(path) for path in selected]
    if not selected or any(path not in available for path in selected):
        raise ValueError("请选择输出目录列表中的视频。")
    if archive.is_symlink() or archive.resolve().is_relative_to(directory.resolve()):
        raise ValueError("ZIP 须保存在 output 外，且不能是软链接。")
    fd, temporary = tempfile.mkstemp(prefix=".minimax-output-", suffix=".zip", dir=archive.parent)
    os.close(fd)
    try:
        with zipfile.ZipFile(temporary, "w", zipfile.ZIP_STORED) as bundle:
            for path in selected:
                if path.is_symlink() or not path.is_file() or not path.resolve().is_relative_to(directory.resolve()):
                    raise ValueError("视频路径已变化，未完成打包。")
                bundle.write(path, arcname=path.relative_to(directory).as_posix())
                if include_workflow:
                    workflow = embedded_workflow(path)
                    if workflow is not None:
                        bundle.writestr("workflow.json", json.dumps(workflow, ensure_ascii=False, indent=2))
        os.replace(temporary, archive)
        return archive
    finally:
        if os.path.exists(temporary):
            os.unlink(temporary)

results = result_files(OUTPUT_DIR)
print("输出目录：", OUTPUT_DIR, "；找到视频：", len(results))
if not results:
    print("暂无视频。生成后重跑本格；已有结果请核对输出目录。")
else:
    selector = widgets.Dropdown(options=[(str(path.relative_to(OUTPUT_DIR)), str(path)) for path in results], description="视频", layout=widgets.Layout(width="95%"))
    button = widgets.Button(description="下载所选视频＋工作流")
    all_button = widgets.Button(description="下载全部视频")
    status = widgets.Output()
    def download(_):
        with status:
            status.clear_output()
            try:
                archive = make_videos_zip([selector.value], OUTPUT_DIR, Path("/content/minimax_result.zip"), include_workflow=True)
                files.download(str(archive))
            except Exception as error:
                print("下载失败：", error)
    def download_all(_):
        with status:
            status.clear_output()
            try:
                archive = make_videos_zip(result_files(OUTPUT_DIR), OUTPUT_DIR, Path("/content/minimax_all_output_videos.zip"))
                print("全部视频已打包。")
                files.download(str(archive))
            except Exception as error:
                print("下载全部视频失败：", error)
    button.on_click(download)
    all_button.on_click(download_all)
    display(selector, widgets.HBox([button, all_button]), status)


## 按需诊断

需要排查或停止服务时运行下方各格。


In [ ]:
# @title 诊断 A：检查模型和磁盘
import os, shutil

COMFY_DIR = "/content/ComfyUI"

files = [
    "models/diffusion_models/minimax_h3_fl2va_pruned_int8_convrot.safetensors",
    "models/text_encoders/qwen3vl_32b_minimax_h3_int8_convrot.safetensors",
    "models/vae/minimax_h3_video_vae_fp16.safetensors",
    "models/vae/minimax_h3_audio_vae_fp32.safetensors",
]

if DOWNLOAD_R2V:
    files.append("models/diffusion_models/minimax_h3_ref2va_pruned_int8_convrot.safetensors")

for rel in files:
    p = os.path.join(COMFY_DIR, rel)
    if os.path.isfile(p):
        print("[OK]", rel, f"{os.path.getsize(p)/1024**3:.2f} GB")
    else:
        print("[MISS]", rel)

total, used, free = shutil.disk_usage("/content")
print(
    f"\n/content disk: used {used/1024**3:.1f} GB / "
    f"total {total/1024**3:.1f} GB / free {free/1024**3:.1f} GB"
)

In [ ]:
# @title 诊断 B：GPU 与 ComfyUI 日志
import subprocess
from pathlib import Path

print(subprocess.run(
    ["nvidia-smi"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
).stdout)

print("\n" + "=" * 80)
print("ComfyUI 日志末尾")
print("=" * 80)

p = Path("/content/comfyui_h3.log")
if p.exists():
    print(p.read_text(errors="replace")[-12000:])
else:
    print("尚未找到日志。")

In [ ]:
# @title 诊断 C：H3 关键加载日志
from pathlib import Path

p = Path("/content/comfyui_h3.log")
patterns = (
    "Requested to load",
    "loaded completely",
    "loaded partially",
    "offload",
    "Prompt executed",
    "model weight dtype",
    "Using async",
)

if p.exists():
    lines = p.read_text(errors="replace").splitlines()
    hits = [
        line for line in lines
        if any(x.lower() in line.lower() for x in patterns)
    ]
    for line in hits[-150:]:
        print(line)
else:
    print("尚未找到日志。")

In [ ]:
# @title 停止 ComfyUI（用完再执行）
STOP_COMFYUI = False # @param {type:"boolean"}

if STOP_COMFYUI:
    import os, signal, time
    pid_path = "/content/comfyui_h3.pid"

    if os.path.exists(pid_path):
        try:
            pid = int(open(pid_path).read().strip())
            os.kill(pid, signal.SIGTERM)
            time.sleep(2)
            print("已停止 ComfyUI PID:", pid)
        except ProcessLookupError:
            print("进程已经不存在。")
    else:
        print("没有找到 PID 文件。")
else:
    print("未停止服务。")